In [ ]:
# Paths come from `paths.py`, never from a literal relative to some checkout.
# That module is the only place that knows where the tree lives, and it names
# the environment variables that override each location (FSCORE_DB above all —
# fscore.db is ~1.7 GB and is not kept in the repository).
# Run this notebook from its own directory, src/fscore_vietnam.
import sys, pathlib

HERE = pathlib.Path.cwd()
assert (HERE / "paths.py").exists(), f"run from src/fscore_vietnam (cwd={HERE})"
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from paths import DATA, RESULTS, DB, ensure_dirs, require_db
ensure_dirs()

# Book-to-market

`BM_INPUTS` in `statement_fields.py` names the four balance-sheet lines. The arithmetic:

    book_equity   = owner_equity - minority_interest
    shares        = shares outstanding at the fiscal year end
    BM            = book_equity / (close_raw * shares)

**Share count.** Charter capital is held at a fixed 10,000 par under VAS, so
`paid_in_capital / PAR_VALUE_VND` converts to a share count exactly — but to the count of
shares *issued*, which includes any the firm has bought back. Market equity wants the
shares outstanding. The share-count section below sources those and says when it trusts
them; `share_count_diagnostics.ipynb` is where the two candidates were measured against
each other.

**Price source.** BM needs the **raw** close: the market value the firm actually carried at
the fiscal year end, against the book value of that same date. The database holds two price
tables and only one of them can answer that:

| table | bar | price |
|---|---|---|
| `simplize_prices` | monthly | adjusted for splits and dividends |
| `fireant_prices` | daily | **raw** `price_close`, with `adj_ratio` kept alongside |

An adjusted close is pushed *below* the raw close of the day by every corporate action that
came after it, so it understates market equity and overstates BM — worst in the oldest
years, which have the most subsequent actions behind them. The last cell measures that gap
on this panel: the adjusted December close is ~4.2x below the raw one in 2009, converging
to 1.0x in 2025. This notebook uses `fireant_prices` and never touches `adj_ratio`; the
adjusted series is for return calculations, not for a point-in-time market value.

In [1]:
import sqlite3
from contextlib import closing

import numpy as np
import pandas as pd

from statement_fields import (BM_INPUTS, MAX_TREASURY_FRAC, PAR_VALUE_VND,
                              SHARE_COLS, SHARE_GAP_TOL, share_count)


PANEL_UNIT_VND = 1.0   # the corrected extract is already in VND — see the note below
FISCAL_END_MONTH = 12  # calendar year end; the panel carries no fiscal-calendar field

# The share-count rule and its two thresholds live in `statement_fields` because
# `tradable_by_turnover_filter.ipynb` divides volume by the same number, and
# `price_matching_and_finalize.ipynb` asserts the two panels agree on it.
BM_INPUTS

(('balance_sheet', 'owner_equity'), ('balance_sheet', 'minority_interest'), ('balance_sheet', 'paid_in_capital'), ('balance_sheet', 'treasury_stock'))

## Inputs

The same universe the F-score runs on: rows of the corrected extract that survived the
accounting checks. Reading `accounting_clean_bol_matrix.csv` only to intersect indexes
keeps this notebook independent of `f_score_calculation.ipynb` while landing on exactly
the same firm-years, so the two panels join on `(symbol, period)` without a coverage
mismatch appearing out of nowhere.

*Units.* The extract is already in VND — HPG's 2024 charter capital reads 6.396e13, i.e.
6.396bn shares at par. `FIREANT.unit_vnd = 1e6` in `statement_fields.py` describes the raw
fireant payload, not this CSV, so `PANEL_UNIT_VND = 1.0` here. Passing `FIREANT` to
`statement_fields.book_to_market()` would inflate book equity a millionfold.

In [2]:
accounting_clean_df = pd.read_csv(f"{RESULTS}/accounting_clean_bol_matrix.csv")
accounting_clean_df.set_index(keys=["symbol", "period"], inplace=True)

extract_df = pd.read_csv(f"{RESULTS}/f_score_fields_extract_corrected.csv")
extract_df.set_index(keys=["symbol", "period"], inplace=True)

BM_KEYS = [key for _, key in BM_INPUTS]
bs = extract_df.loc[extract_df.index.isin(accounting_clean_df.index), BM_KEYS].sort_index()
bs

owner_equity  minority_interest  paid_in_capital  treasury_stock
symbol period                                                                  
A32    2017    1.763000e+11                0.0     6.800000e+10             0.0
       2018    2.008266e+11                0.0     6.800000e+10             0.0
       2019    2.236159e+11                0.0     6.800000e+10             0.0
       2020    2.422229e+11                0.0     6.800000e+10             0.0
       2021    2.380572e+11                0.0     6.800000e+10             0.0
...                     ...                ...              ...             ...
YTC    2021    2.174660e+10                0.0     3.080000e+10             0.0
       2022    3.240305e+10                0.0     3.080000e+10             0.0
       2023    3.897662e+10                0.0     3.080000e+10             0.0
       2024    1.796766e+11                0.0     9.548000e+10             0.0
       2025    1.905966e+11                0.0     9.548000e+10             0.0

[21233 rows x 4 columns]

## Price: daily bars normalised to month ends

`fireant_prices` is a daily table, so it needs collapsing to one observation per calendar
month before anything can be matched to an annual panel. Three rules, all visible in the
SQL below:

1. **`price_close * unit`.** The stored price is in the row's own `unit` — 1,000 for
   shares, 1 for the five indices. Multiplying gives VND, and the `unit = 1000` filter is
   also what keeps VNINDEX and friends out of a table keyed by ticker.
2. **Last *traded* session of the month.** 38% of the daily bars have `total_volume = 0` —
   real non-trading sessions on UPCoM, not missing data. Their `price_close` is the
   reference price carried over from the last trade, so ordering by `total_volume > 0`
   first takes the last session where a price was actually paid.
3. **Months with no trade at all still get a bar**, flagged `stale`. Dropping them would
   silently delist every illiquid firm rather than mark it, and the flag lets a later
   screen decide. It changes 458 of the December closes used here, by a median 1.1%.

In [3]:
# One row per (symbol, month): the last traded session of that month, in VND.
# `sessions` / `traded_sessions` are carried so the price can be judged, not just used.
MONTH_END_SQL = """
select symbol, month, date, close, sessions, traded_sessions
from (
    select symbol,
           substr(date, 1, 7)                    as month,
           date,
           price_close * unit                    as close,
           count(*)              over w          as sessions,
           sum(total_volume > 0) over w          as traded_sessions,
           row_number()          over (partition by symbol, substr(date, 1, 7)
                                       order by total_volume > 0 desc, date desc) as rn
    from fireant_prices
    where unit = 1000 and price_close > 0
    window w as (partition by symbol, substr(date, 1, 7))
)
where rn = 1
"""

with closing(sqlite3.connect(DB)) as conn:
    month_end = pd.read_sql(MONTH_END_SQL, conn)

month_end["stale"] = month_end["traded_sessions"].eq(0)
month_end["period"] = month_end["month"].str[:4].astype(int)
month_end = month_end.set_index(["symbol", "month"]).sort_index()
month_end

date    close  sessions  traded_sessions  stale  period
symbol month                                                                 
A32    2018-10  2018-10-31  25900.0         7                0   True    2018
       2018-11  2018-11-29  32000.0        22                1  False    2018
       2018-12  2018-12-21  30200.0        20                3  False    2018
       2019-01  2019-01-07  32000.0        22                2  False    2019
       2019-02  2019-02-19  22000.0        15                3  False    2019
...                    ...      ...       ...              ...    ...     ...
YTC    2026-04  2026-04-28  24000.0        20                4  False    2026
       2026-05  2026-05-29  24000.0        20                0   True    2026
       2026-06  2026-06-19  25500.0        22                2  False    2026
       2026-07  2026-07-14  21700.0        23                1  False    2026
       2026-08  2026-08-07  21700.0         5                0   True    2026

[241632 rows x 6 columns]

In [4]:
months = month_end.index.get_level_values("month")

pd.Series({
    "month-end bars": len(month_end),
    "symbols": month_end.index.get_level_values("symbol").nunique(),
    "span": f"{months.min()} .. {months.max()}",
    "months with no trade at all (stale close)": int(month_end["stale"].sum()),
    "  as a share of all month-ends": f'{100 * month_end["stale"].mean():.1f}%',
    "median traded sessions per month": int(month_end["traded_sessions"].median()),
})

month-end bars                                           241632
symbols                                                    1824
span                                         2009-01 .. 2026-08
months with no trade at all (stale close)                 32030
  as a share of all month-ends                            13.3%
median traded sessions per month                             15
dtype: object

## Matching to the fiscal year end

The panel is annual, so the December month-end is the market value that faces the balance
sheet of the same `period`. Nothing is rolled back from an earlier month: the next cell
shows why there is almost nothing to roll back to — the unmatched rows are overwhelmingly
firm-years filed *before* the stock ever traded, and no price existed to fetch.

In [5]:
def period_end_close(month_end: pd.DataFrame, index: pd.MultiIndex,
                     fiscal_end_month: int = FISCAL_END_MONTH) -> pd.DataFrame:
    """
    The month-end close at each firm's fiscal year end, aligned onto the panel index.

    Returns the date the price came from and the stale flag alongside it: a BM built on a
    23 December print of an untraded stock and one built on the 31 December close of a
    liquid one are not the same measurement, and the panel should say which it is.
    """
    month = month_end.index.get_level_values("month")
    at_end = month_end[month.str.endswith(f"-{fiscal_end_month:02d}")]
    at_end = at_end.reset_index().set_index(["symbol", "period"])
    return pd.DataFrame({
        "close": at_end["close"].reindex(index),
        "close_date": at_end["date"].reindex(index),
        "price_stale": at_end["stale"].reindex(index).astype("boolean"),
    })


px = period_end_close(month_end, bs.index)
px

close  close_date  price_stale
symbol period                                   
A32    2017         NaN         NaN         <NA>
       2018     30200.0  2018-12-21        False
       2019     28000.0  2019-12-24        False
       2020     34500.0  2020-12-31        False
       2021     31800.0  2021-12-31        False
...                 ...         ...          ...
YTC    2021     55000.0  2021-12-31        False
       2022    110000.0  2022-12-28        False
       2023    136000.0  2023-12-29        False
       2024     40000.0  2024-12-27        False
       2025     29500.0  2025-12-05        False

[21233 rows x 3 columns]

In [6]:
# Where the unmatched firm-years come from. A missing price is not a data defect when the
# stock was not listed yet: those rows have no market value to measure, and the only fix
# is a longer price history, which does not exist.
traded_year = pd.Series(months.str[:4].astype(int),
                        index=month_end.index.get_level_values("symbol"))
span = traded_year.groupby(level=0).agg(["min", "max"])

symbol = bs.index.get_level_values("symbol")
period = bs.index.get_level_values("period")
first = span["min"].reindex(symbol).to_numpy()
last = span["max"].reindex(symbol).to_numpy()
unmatched = px["close"].isna().to_numpy()

pd.Series({
    "panel rows": len(bs),
    "matched to a close": int((~unmatched).sum()),
    "unmatched": int(unmatched.sum()),
    "  filed before the stock first traded": int((unmatched & (period < first)).sum()),
    "  filed after it last traded": int((unmatched & (period > last)).sum()),
    "  gap inside the traded span": int((unmatched & (period >= first) & (period <= last)).sum()),
})

panel rows                               21233
matched to a close                       18086
unmatched                                 3147
  filed before the stock first traded     3041
  filed after it last traded                11
  gap inside the traded span                95
dtype: int64

## Share count: issued, outstanding, and which one the denominator gets

`paid_in_capital / PAR_VALUE_VND` counts every share ever registered, treasury included.
Market equity wants the shares actually in the market, and `book_equity` above has already
netted treasury out — VAS carries it negative inside `owner_equity`. Leaving the
denominator gross therefore mixes two standards in one ratio.

`fireant_financial_data_general.ShareAtPeriodEnd` supplies the outstanding count, and it is
not usable raw. `share_count_diagnostics.ipynb` finds 826 firm-years where it exceeds the
issued count, which cannot happen, and 140 symbols where it is frozen at the crawl-date
value across their entire history — today's share count back-filled onto 2011, i.e.
look-ahead straight into the panel. Two guards, read off one signed column:

| test | reading |
|---|---|
| `shares_gap > +0.5%` | outstanding above issued — impossible, so vendor error |
| `shares_gap < -30%` | past any believable buyback; the frozen-count failure lands here |

Rows failing either guard fall back to the par-implied count, which is what this notebook
used before, so the change costs no BM coverage. Inside the `+0.5%` tolerance the vendor
count is used but clipped at the issued count: outstanding above issued is noise wherever
it is small and error wherever it is large, and neither is a share count.

**The fix is partial, and the write-up should say so.** Of the firm-years whose balance
sheet records treasury stock, the vendor nets it out on 90.1%; on the remainder
`ShareAtPeriodEnd` simply *is* the issued count and the old bias survives. That is why
`has_treasury` stays in the export.

In [7]:
SHARE_SQL = """
select symbol, Year as period, ShareAtPeriodEnd
from fireant_financial_data_general
where Quarter = 0 and ShareAtPeriodEnd > 0
"""

with closing(sqlite3.connect(DB)) as conn:
    vendor_shares = pd.read_sql(SHARE_SQL, conn).set_index(["symbol", "period"])["ShareAtPeriodEnd"]

shares = share_count(bs["paid_in_capital"], vendor_shares, unit_vnd=PANEL_UNIT_VND)
shares

shares_issued  shares_out  shares_gap shares_source     shares
symbol period                                                                
A32    2017        6800000.0   6800000.0         0.0   outstanding  6800000.0
       2018        6800000.0   6800000.0         0.0   outstanding  6800000.0
       2019        6800000.0   6800000.0         0.0   outstanding  6800000.0
       2020        6800000.0   6800000.0         0.0   outstanding  6800000.0
       2021        6800000.0   6800000.0         0.0   outstanding  6800000.0
...                      ...         ...         ...           ...        ...
YTC    2021        3080000.0   3080000.0         0.0   outstanding  3080000.0
       2022        3080000.0   3080000.0         0.0   outstanding  3080000.0
       2023        3080000.0   3080000.0         0.0   outstanding  3080000.0
       2024        9548000.0   9548000.0         0.0   outstanding  9548000.0
       2025        9548000.0   9548000.0         0.0   outstanding  9548000.0

[21233 rows x 5 columns]

In [8]:
# Where the denominator came from, and why it fell back when it did.
gap = shares["shares_gap"]
pd.Series({
    "panel rows": len(shares),
    "denominator = ShareAtPeriodEnd": int(shares["shares_source"].eq("outstanding").sum()),
    "  identical to the issued count": int(gap.abs().le(SHARE_GAP_TOL).sum()),
    "  strictly below it (treasury netted)": int(gap.lt(-SHARE_GAP_TOL).sum()),
    "fell back to par-implied": int(shares["shares_source"].eq("par_implied").sum()),
    "  no vendor count": int(shares["shares_out"].isna().sum()),
    "  outstanding > issued": int(gap.gt(SHARE_GAP_TOL).sum()),
    f"  gap below -{MAX_TREASURY_FRAC:.0%}": int(gap.lt(-MAX_TREASURY_FRAC).sum()),
})

panel rows                               21233
denominator = ShareAtPeriodEnd           20415
  identical to the issued count          18010
  strictly below it (treasury netted)     2539
fell back to par-implied                   818
  no vendor count                          196
  outstanding > issued                     483
  gap below -30%                           134
dtype: int64

## The ratio

What it adjusts for, and what it deliberately leaves alone:

- **`treasury_stock` is not subtracted from book equity.** Under VAS it is already carried
  negative inside `owner_equity`; netting it again would double-count.
- **Treasury is netted out of the denominator instead**, through `shares` above. This is
  the half that used to be missing: book equity was net of treasury while market equity was
  gross of it, which pushed BM down for exactly the firms that ran a buyback. `has_treasury`
  is still exported, because the vendor's netting is only 90.1% complete.
- **Minority interest is subtracted**, so book equity is the parent's, matching
  `net_income_parent` in the F-score.
- **Negative book equity yields NaN, not a number.** A negative book value does not make a
  firm cheap; left in, it sorts straight to the top of a value screen.

`bm_shares_issued` is exported next to `bm`: the ratio this notebook produced before the
share-count change, kept so downstream work can measure the switch instead of
reconstructing it.

In [9]:
def _div(num: pd.Series, den: pd.Series) -> pd.Series:
    """num/den with a zero denominator yielding NaN rather than inf."""
    return num / den.replace(0, np.nan)


def book_to_market(bs: pd.DataFrame, px: pd.DataFrame, shares: pd.DataFrame,
                   unit_vnd: float = PANEL_UNIT_VND) -> pd.DataFrame:
    """BM at the fiscal year end, plus every intermediate it is built from."""
    out = pd.DataFrame(index=bs.index)
    out["book_equity"] = (bs["owner_equity"] - bs["minority_interest"].fillna(0)) * unit_vnd
    out[SHARE_COLS] = shares[SHARE_COLS]
    out["close"] = px["close"]
    out["market_equity"] = out["close"] * out["shares"]

    positive = out["book_equity"] > 0
    out["bm"] = _div(out["book_equity"], out["market_equity"]).where(positive)
    out["bm_shares_issued"] = _div(out["book_equity"],
                                   out["close"] * out["shares_issued"]).where(positive)

    out["close_date"] = px["close_date"]
    out["price_stale"] = px["price_stale"]
    out["has_treasury"] = bs["treasury_stock"].fillna(0).ne(0)
    return out


bm = book_to_market(bs, px, shares)
bm

book_equity  shares_issued  ...  price_stale  has_treasury
symbol period                               ...                           
A32    2017    1.763000e+11      6800000.0  ...         <NA>         False
       2018    2.008266e+11      6800000.0  ...        False         False
       2019    2.236159e+11      6800000.0  ...        False         False
       2020    2.422229e+11      6800000.0  ...        False         False
       2021    2.380572e+11      6800000.0  ...        False         False
...                     ...            ...  ...          ...           ...
YTC    2021    2.174660e+10      3080000.0  ...        False         False
       2022    3.240305e+10      3080000.0  ...        False         False
       2023    3.897662e+10      3080000.0  ...        False         False
       2024    1.796766e+11      9548000.0  ...        False         False
       2025    1.905966e+11      9548000.0  ...        False         False

[21233 rows x 13 columns]

### Coverage and diagnostics

In [10]:
has_bm = bm["bm"].notna()
from_vendor = bm["shares_source"].eq("outstanding")

pd.Series({
    "panel rows": len(bm),
    "with BM": int(has_bm.sum()),
    "  no close at the fiscal year end": int(bm["close"].isna().sum()),
    "  no charter capital": int((bm["shares_issued"] == 0).sum()),
    "  book equity <= 0": int((bm["book_equity"] <= 0).sum()),
    "priced off a month with no trade": int(bm["price_stale"].fillna(False).sum()),
    "treasury stock held": int(bm["has_treasury"].sum()),
    "denominator from ShareAtPeriodEnd": int((has_bm & from_vendor).sum()),
    "  and strictly below the issued count": int((has_bm & from_vendor
                                                  & bm["shares_gap"].lt(-SHARE_GAP_TOL)).sum()),
    "denominator fell back to par-implied": int((has_bm & ~from_vendor).sum()),
})

panel rows                               21233
with BM                                  17414
  no close at the fiscal year end         3147
  no charter capital                         5
  book equity <= 0                         736
priced off a month with no trade          2166
treasury stock held                       5109
denominator from ShareAtPeriodEnd        17013
  and strictly below the issued count     2129
denominator fell back to par-implied       401
dtype: int64

In [11]:
# What the share-count change did. The direction is one-sided by construction: netting
# treasury out of the denominator shrinks market equity, so BM can only rise.
moved = (bm["bm"] / bm["bm_shares_issued"]).dropna()
material = moved.sub(1).abs().gt(0.01)

pd.Series({
    "rows with BM": int(has_bm.sum()),
    "BM unchanged": int(moved.sub(1).abs().le(1e-9).sum()),
    "BM changed > 1%": f"{int(material.sum())} ({100 * material.mean():.1f}%)",
    "BM changed > 5%": int(moved.sub(1).abs().gt(0.05).sum()),
    "median change among those that moved": f"{moved[material].median():.4f}x",
    "p95 among those that moved": f"{moved[material].quantile(0.95):.4f}x",
    "largest": f"{moved.max():.4f}x",
})

rows with BM                                   17414
BM unchanged                                   12917
BM changed > 1%                         1793 (10.3%)
BM changed > 5%                                  744
median change among those that moved         1.0387x
p95 among those that moved                   1.1653x
largest                                      1.4286x
dtype: object

In [12]:
# Level and coverage by year. The median sits near 1.0 throughout — no drift, which is the
# point of using a raw price series: the drift in the adjusted one is an artefact of the
# adjustment, not a re-rating of the market. `stale_pct` rises with the UPCoM cohort after
# 2016 and is the number to watch before treating a year's cross-section as tradeable.
pd.DataFrame({
    "rows": has_bm.groupby(level="period").size(),
    "with_bm": has_bm.groupby(level="period").sum(),
    "median_bm": bm["bm"].groupby(level="period").median().round(3),
    "p25": bm["bm"].groupby(level="period").quantile(0.25).round(3),
    "p75": bm["bm"].groupby(level="period").quantile(0.75).round(3),
    "stale_pct": (100 * bm["price_stale"].fillna(False).groupby(level="period").sum()
                  / has_bm.groupby(level="period").sum()).round(1),
})

,rows,with_bm,median_bm,p25,p75,stale_pct
period,,,,,,
2009,525,274,0.697,0.475,0.973,0.4
2010,751,544,0.919,0.673,1.195,1.7
2011,875,672,1.918,1.272,2.887,5.7
2012,984,719,1.813,1.224,2.632,7.6
2013,1060,708,1.522,1.021,2.321,7.1
2014,1111,713,1.222,0.807,1.821,5.3
2015,1255,799,1.132,0.766,1.766,6.9
2016,1369,941,1.094,0.698,1.798,13.5
2017,1482,1232,1.063,0.691,1.650,16.2


In [13]:
# What the adjusted series would have done to this panel, year by year. This is the whole
# case for the source switch: using `simplize_prices` divides market equity by the numbers
# in `median_ratio`, i.e. multiplies BM by ~4.2x in 2009 and ~1.0x in 2025 — a spurious
# value premium in exactly the direction a long-sample backtest would reward.
with closing(sqlite3.connect(DB)) as conn:
    simplize_dec = pd.read_sql(
        "select symbol, time, close from simplize_prices "
        "where resolution = 'monthly' and time like '%-12-01 %'", conn)

simplize_dec["period"] = simplize_dec["time"].str[:4].astype(int)
simplize_dec = simplize_dec.set_index(["symbol", "period"])["close"]

price_check = pd.DataFrame({
    "fireant_raw": bm["close"],
    "simplize_adjusted": simplize_dec.reindex(bm.index),
}).dropna()
price_check["ratio"] = price_check["fireant_raw"] / price_check["simplize_adjusted"]

price_check.groupby(level="period")["ratio"].agg(rows="size", median_ratio="median").round(3)

,rows,median_ratio
period,,
2009,255,4.237
2010,491,3.205
2011,601,2.782
2012,649,2.454
2013,662,2.222
2014,674,2.014
2015,773,1.856
2016,906,1.672
2017,1175,1.561


In [14]:
# Spot check against reality: these three market caps are public numbers.
bm.loc[[("HPG", 2024), ("VNM", 2024), ("FPT", 2024)],
       ["close", "shares_issued", "shares", "shares_source", "market_equity",
        "book_equity", "bm"]]

,,close,shares_issued,shares,shares_source,market_equity,book_equity,bm
symbol,period,,,,,,,
HPG,2024,26650.0,6.396250e+09,6.396250e+09,outstanding,1.704601e+14,1.143565e+14,0.670870
VNM,2024,63400.0,2.089955e+09,2.089955e+09,outstanding,1.325032e+14,3.227882e+13,0.243608
FPT,2024,152500.0,1.471069e+09,1.471069e+09,outstanding,2.243381e+14,2.979148e+13,0.132797


### Export

The intermediates travel with the ratio. `close_date`, `price_stale` and `has_treasury` are
what let a later screen decide whether a given BM is tradeable; `shares_out`, `shares_gap`
and `shares_source` are what let it decide whether the denominator is the vendor's number
or the par-implied fallback. A bare `bm` column cannot be audited once it is three
notebooks downstream.

`shares_issued` keeps its name and its old meaning — `price_matching_and_finalize.ipynb`
asserts it against the turnover panel's, which derives from the same charter capital.

Keyed on `(symbol, period)`, so it joins straight onto `f_score_panel.csv`.

In [15]:
bm.to_csv(f"{RESULTS}/book_to_market_panel.csv")
bm.shape

(21233, 13)